## Notebook to split all liganded human PDB into target/seed

Split all human liganded pdb into:
- target: ligase+glue complex (e.g. VHL, CRBN)
- seed: database of any liganded human PDB structure

### Input:
- `human_reference_proteome_pdb_ligands.csv` from git@github.com:meng-yx/ligand_viewer.git repo

In [5]:
import os
import pandas as pd
from pathlib import Path

repo_root = !git rev-parse --show-toplevel
repo_root = repo_root[0]
os.chdir(repo_root)

data_dir = Path(repo_root) / "data/human_reference_proteome_liganded_pdbs"
os.makedirs(data_dir, exist_ok=True)

# ----- Input ------
human_ref_proteome_dir = Path("/work/upthomae/Meng/ligand_viewer/data/human_reference_proteome/")
all_ligands_csv = os.path.join(human_ref_proteome_dir, "human_reference_proteome_drug_like_fragments.csv")

# ----- Output ------
all_ligands_split_csv = os.path.join(data_dir, "human_reference_proteome_pdb_ligands_split.csv")

In [12]:
df_all_ligands = pd.read_csv(all_ligands_csv)
print(f"df_all_ligands.shape: {df_all_ligands.shape}")

df_all_ligands.head()

df_all_ligands.shape: (15603, 19)


,uniprot_id,gene_name,recommendedName,pdb_id,protein_chain,ligand_chain,ligand_code,ligand_name,smiles,formula,mw,qed,num_carbon,num_N_O,uniprot_id_count,method,resolution,neighbor_count,percent_intracellular
0,Q8TBX8,PIP4K2C,Phosphatidylinositol 5-phosphate 4-kinase type...,7QPN,B,B,DVF,5-methyl-2-(2-propan-2-ylphenyl)-~{N}-(pyridin...,CC(C)c1ccccc1c2nc3ccn(c3c(n2)NCc4ccccn4)C,C22H23N5,357.195346,0.559088,22.0,5.0,1.0,X-RAY DIFFRACTION,1.95,32.0,1.0
1,Q15562,TEAD2,Transcriptional enhancer factor TEF-4,8CUH,B,B,P0I,4-[3-(2-cyclohexylethoxy)benzoyl]-N-phenylpipe...,c1ccc(cc1)NC(=O)N2CCN(CC2)C(=O)c3cccc(c3)OCCC4...,C26H33N3O3,435.252192,0.692559,26.0,6.0,1.0,X-RAY DIFFRACTION,2.40,28.0,1.0
2,Q07869,PPARA,Peroxisome proliferator-activated receptor alpha,2P54,A,A,735,2-METHYL-2-(4-{[({4-METHYL-2-[4-(TRIFLUOROMETH...,Cc1c(sc(n1)c2ccc(cc2)C(F)(F)F)C(=O)NCc3ccc(cc3...,C23H21F3N2O4S,478.117413,0.480598,23.0,6.0,1.0,X-RAY DIFFRACTION,1.79,28.0,1.0
3,P62508,ESRRG,Estrogen-related receptor gamma,6A6K,B,B,9S6,3-[(~{E})-5-oxidanyl-2-phenyl-1-[4-(4-propan-2...,CC(C)N1CCN(CC1)c2ccc(cc2)/C(=C(/CCCO)\c3ccccc3...,C30H36N2O2,456.277678,0.429883,30.0,4.0,1.0,X-RAY DIFFRACTION,2.90,27.0,1.0
4,Q07869,PPARA,Peroxisome proliferator-activated receptor alpha,6KB0,A,A,ITY,"icosa-5,8,11,14-tetraynoic acid",CCCCCC#CCC#CCC#CCC#CCCCC(=O)O,C20H24O2,296.177630,0.593853,20.0,2.0,1.0,X-RAY DIFFRACTION,1.35,27.0,1.0


In [13]:
# Filter to only X-ray DIFFRACTION structures
df_all_ligands = df_all_ligands[df_all_ligands['method'] == 'X-RAY DIFFRACTION']
print(f"df_all_ligands.shape: {df_all_ligands.shape}")


df_all_ligands.shape: (15233, 19)


In [14]:
# List of known glueable ligases
uniprot_ligases = [
    "Q96SW2", # CRBN
    "P40337", # VHL
    "Q66K64", # DCAF15
    "P41182", # BCL6
    "Q9Y297", # FBW1A
    "Q9NXF7", # DCAF16
    "P62942", # FKBP12 
    "Q9NVX7"  # KBTBD4
]

df_all_ligands["split"] = df_all_ligands["uniprot_id"].apply(lambda x: "target" if x in uniprot_ligases else "seed")

df_all_ligands.to_csv(all_ligands_split_csv, index=False)

df_all_ligands[df_all_ligands["split"] == "target"].head()

,uniprot_id,gene_name,recommendedName,pdb_id,protein_chain,ligand_chain,ligand_code,ligand_name,smiles,formula,mw,qed,num_carbon,num_N_O,uniprot_id_count,method,resolution,neighbor_count,percent_intracellular,split
2322,P62942,FKBP1A,Peptidyl-prolyl cis-trans isomerase FKBP1A,3MDY,C,C,LDN,"4-[6-(4-piperazin-1-ylphenyl)pyrazolo[1,5-a]py...",c1ccc2c(c1)c(ccn2)c3cnn4c3ncc(c4)c5ccc(cc5)N6C...,C25H22N6,406.190595,0.490424,25.0,6.0,3.0,X-RAY DIFFRACTION,2.05,18.0,1.0,target
3841,P40337,VHL,von Hippel-Lindau disease tumor suppressor,9QE4,I,I,A1I57,"(2~{S},4~{R})-1-[(2~{S})-2-[(1-fluoranylcyclop...",Cc1c(scn1)c2ccc(cc2)[C@H](C)NC(=O)[C@@H]3C[C@H...,C27H35FN4O4S,530.236305,0.508790,27.0,8.0,1.0,X-RAY DIFFRACTION,2.28,16.0,1.0,target
3842,P40337,VHL,von Hippel-Lindau disease tumor suppressor,9QE5,C,C,A1I58,"(2~{S},4~{R})-1-[(2~{S})-1-(1-fluoranylcyclopr...",Cc1c(scn1)c2ccc(cc2)[C@H](C)NC(=O)[C@@H]3C[C@H...,C27H33FN4O4S,528.220655,0.599852,27.0,8.0,1.0,X-RAY DIFFRACTION,2.50,16.0,1.0,target
4251,P62942,FKBP1A,Peptidyl-prolyl cis-trans isomerase FKBP1A,9LYG,A,A,A1L7S,5-[(2~{S})-1-cyclohexylsulfonylpiperidin-2-yl]...,COc1ccc(cc1OC)CCCc2nc(on2)[C@@H]3CCCCN3S(=O)(=...,C24H35N3O5S,477.229742,0.526401,24.0,8.0,1.0,X-RAY DIFFRACTION,1.26,16.0,1.0,target
5020,Q96SW2,CRBN,Protein cereblon,8RQC,D,D,QFC,Mezigdomide,c1cc2c(c(c1)OCc3ccc(cc3)CN4CCN(CC4)c5ccc(cc5F)...,C32H30FN5O4,567.228183,0.436724,32.0,9.0,2.0,X-RAY DIFFRACTION,2.15,15.0,1.0,target


In [15]:
df_all_ligands[df_all_ligands["split"] == "seed"]

,uniprot_id,gene_name,recommendedName,pdb_id,protein_chain,ligand_chain,ligand_code,ligand_name,smiles,formula,mw,qed,num_carbon,num_N_O,uniprot_id_count,method,resolution,neighbor_count,percent_intracellular,split
0,Q8TBX8,PIP4K2C,Phosphatidylinositol 5-phosphate 4-kinase type...,7QPN,B,B,DVF,5-methyl-2-(2-propan-2-ylphenyl)-~{N}-(pyridin...,CC(C)c1ccccc1c2nc3ccn(c3c(n2)NCc4ccccn4)C,C22H23N5,357.195346,0.559088,22.0,5.0,1.0,X-RAY DIFFRACTION,1.95,32.0,1.0,seed
1,Q15562,TEAD2,Transcriptional enhancer factor TEF-4,8CUH,B,B,P0I,4-[3-(2-cyclohexylethoxy)benzoyl]-N-phenylpipe...,c1ccc(cc1)NC(=O)N2CCN(CC2)C(=O)c3cccc(c3)OCCC4...,C26H33N3O3,435.252192,0.692559,26.0,6.0,1.0,X-RAY DIFFRACTION,2.40,28.0,1.0,seed
2,Q07869,PPARA,Peroxisome proliferator-activated receptor alpha,2P54,A,A,735,2-METHYL-2-(4-{[({4-METHYL-2-[4-(TRIFLUOROMETH...,Cc1c(sc(n1)c2ccc(cc2)C(F)(F)F)C(=O)NCc3ccc(cc3...,C23H21F3N2O4S,478.117413,0.480598,23.0,6.0,1.0,X-RAY DIFFRACTION,1.79,28.0,1.0,seed
3,P62508,ESRRG,Estrogen-related receptor gamma,6A6K,B,B,9S6,3-[(~{E})-5-oxidanyl-2-phenyl-1-[4-(4-propan-2...,CC(C)N1CCN(CC1)c2ccc(cc2)/C(=C(/CCCO)\c3ccccc3...,C30H36N2O2,456.277678,0.429883,30.0,4.0,1.0,X-RAY DIFFRACTION,2.90,27.0,1.0,seed
4,Q07869,PPARA,Peroxisome proliferator-activated receptor alpha,6KB0,A,A,ITY,"icosa-5,8,11,14-tetraynoic acid",CCCCCC#CCC#CCC#CCC#CCCCC(=O)O,C20H24O2,296.177630,0.593853,20.0,2.0,1.0,X-RAY DIFFRACTION,1.35,27.0,1.0,seed
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15598,P03372,ESR1,Estrogen receptor,8BZT,B,A,SKR,"~{N}-[(2~{S})-2-[(1~{E},3~{R},4~{S},8~{R},9~{R...",C[C@@H]1[C@@H]\2CC[C@@H](/C2=C/[C@]3([C@H](CC(...,C23H37NO5,407.267173,0.521047,23.0,6.0,1.0,X-RAY DIFFRACTION,1.65,1.0,1.0,seed
15599,P03372,ESR1,Estrogen receptor,8BZW,B,A,GEH,2-(4-chloranylphenoxy)-2-methyl-~{N}-(2-sulfan...,CC(C)(C(=O)NCCS)Oc1ccc(cc1)Cl,C12H16ClNO2S,273.059027,0.809469,12.0,3.0,1.0,X-RAY DIFFRACTION,1.10,1.0,1.0,seed
15600,P03372,ESR1,Estrogen receptor,8C04,B,A,GEH,2-(4-chloranylphenoxy)-2-methyl-~{N}-(2-sulfan...,CC(C)(C(=O)NCCS)Oc1ccc(cc1)Cl,C12H16ClNO2S,273.059027,0.809469,12.0,3.0,1.0,X-RAY DIFFRACTION,1.10,1.0,1.0,seed
15601,P03372,ESR1,Estrogen receptor,8C0L,B,A,FC7,Fusicoccin A-THF,C[C@@H]1[C@@H]2CC[C@@H](C2=C[C@]3([C@@H]4[C@@H...,C32H50O9,578.345483,0.423661,32.0,9.0,2.0,X-RAY DIFFRACTION,1.60,1.0,1.0,seed
